<a href="https://colab.research.google.com/github/exponen-agi/hybrid-search-rag/blob/main/RAG_%2B_Hybrid_Search_with_Crew_AI%2C_NeonDb%2C_Qdrant_and_Gemini_a_real_world_scenario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Hybrid Search RAG: Recipe Recommender

This notebook builds a small **recipe recommender** to teach one idea: **hybrid search**.

Hybrid search means combining two kinds of search into one result list:

- **Semantic (dense vector) search** — finds recipes that *mean* the same thing as your
  query, even if the words are different. Powered by Google Gemini embeddings.
- **Keyword (sparse vector / TF‑IDF) search** — finds recipes that share the *exact words*
  in your query. Classic full‑text search.

Qdrant fuses both result lists with **Reciprocal Rank Fusion (RRF)**, so you get results
that are both relevant in meaning *and* contain the words you typed. A CrewAI agent then
turns the raw search results into a friendly recommendation using a Gemini LLM.

For the full architecture picture (with diagrams), see **[docs/index.html](docs/index.html)**.

```
                    ┌───────────────────────┐
                    │      Your query        │
                    │ "warm soup for winter" │
                    └───────────┬─────────────┘
                                │
                 ┌──────────────┴───────────────┐
                 ▼                               ▼
      ┌─────────────────────┐        ┌─────────────────────┐
      │ Dense (semantic)     │        │ Sparse (keyword)     │
      │ Gemini embedding     │        │ TF-IDF vector        │
      └──────────┬───────────┘        └──────────┬───────────┘
                 │                                │
                 ▼                                ▼
         ┌───────────────────────────────────────────────┐
         │              Qdrant hybrid query                │
         │         (Reciprocal Rank Fusion / RRF)           │
         └───────────────────────┬───────────────────────┘
                                 ▼
                     ┌───────────────────────┐
                     │  CrewAI "Recipe Expert" │
                     │  agent (Gemini LLM)      │
                     └───────────┬─────────────┘
                                 ▼
                    Friendly recipe recommendation
```


## Two ways to run this notebook

You do **not** need any cloud accounts to try this out. There are two modes, controlled by
one flag in the next code cell (`TEST_MODE`):

| | `TEST_MODE = True` (default) | `TEST_MODE = False` |
|---|---|---|
| Vector database | In‑memory Qdrant (nothing to install) | Your Qdrant Cloud cluster |
| Recipe storage | Plain Python list, in memory | Your Neon/PostgreSQL database |
| Embeddings | Deterministic "fake" embeddings (no API key, no cost) | Real Gemini embeddings (needs `GEMINI_API_KEY`) |
| Final answer | The raw hybrid search results (no LLM call) | A CrewAI agent writes a friendly recommendation (needs `GEMINI_API_KEY`) |
| Good for | Learning the code, CI smoke tests, offline demos | The real, end-to-end product experience |

Start with `TEST_MODE = True` to see the whole pipeline run in a few seconds. Then switch to
`TEST_MODE = False` and add your API keys once you want the real Gemini-powered experience.

### Prerequisites for `TEST_MODE = False`
* A Google AI Studio / Gemini API key ([get one here](https://aistudio.google.com/app/apikey))
* A free [Neon](https://neon.tech) PostgreSQL database
* A free [Qdrant Cloud](https://cloud.qdrant.io) cluster
* Python 3.9+


In [ ]:
# Install the exact versions this notebook was tested with.
# (See requirements.txt in the repo root if you are running outside Colab.)
%pip install -q \
    "crewai==1.15.18" \
    "crewai-tools==1.15.18" \
    "qdrant-client==1.19.0" \
    "langchain-google-genai==4.4.0" \
    "psycopg2-binary==2.9.12" \
    "scikit-learn==1.9.0" \
    "faker==40.38.0"

In [ ]:
import hashlib
import os
import time

import numpy as np
from qdrant_client import QdrantClient, models

# --- Run mode -----------------------------------------------------------
# True  -> fully offline demo: in-memory Qdrant, fake embeddings, no API keys.
# False -> the real pipeline: Neon + Qdrant Cloud + Gemini (needs credentials below).
TEST_MODE = True

# --- Credentials (only required when TEST_MODE = False) -----------------
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

DB_HOST = os.environ.get("DB_HOST", "postgres host")
DB_NAME = os.environ.get("DB_NAME", "database name")
DB_USER = os.environ.get("DB_USER", "db user name")
DB_PASSWORD = os.environ.get("DB_PASSWORD", "db user password")

QDRANT_URL = os.environ.get("QDRANT_URL", "")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY", "")

# --- Fixed settings -------------------------------------------------------
COLLECTION_NAME = "food_recipes"
# `gemini-embedding-001` is being retired (Google has it scheduled for shutdown
# between Jul-Oct 2026), so this notebook uses its successor, `gemini-embedding-2`,
# which is generally available and keeps the same 3072-dim default output. If you
# still have access to `models/gemini-embedding-001` and prefer it, it's a drop-in
# swap for this one line.
EMBEDDING_MODEL_NAME = "models/gemini-embedding-2"
EMBEDDING_DIM = 3072  # gemini-embedding-2's default output size (128-3072, configurable)
GEMINI_LLM_MODEL = "gemini/gemini-3.5-flash"  # current stable Gemini Flash tier

if not TEST_MODE and not GEMINI_API_KEY:
    raise RuntimeError(
        "TEST_MODE is False but GEMINI_API_KEY is not set. "
        "Either set TEST_MODE = True to try the offline demo, "
        "or export GEMINI_API_KEY (and the DB_*/QDRANT_* variables) first."
    )

print(f"Running in {'TEST' if TEST_MODE else 'LIVE'} mode.")

## Embeddings and the vector database client

`get_embedding_model()` returns something with an `.embed_query(text)` method, exactly like
the real `GoogleGenerativeAIEmbeddings` class. In test mode we return a tiny stand‑in that
turns text into a **deterministic** vector using a hash — same input text always gives the
same vector, no network call, no API key. That is enough to prove the hybrid‑search plumbing
works; it does not need to produce *meaningful* embeddings.

`get_qdrant_client()` gives you either a local, in‑memory Qdrant (`:memory:`) or a real
Qdrant Cloud connection, depending on `TEST_MODE`.


In [3]:
class FakeEmbeddings:
    """Deterministic, offline stand-in for GoogleGenerativeAIEmbeddings.

    Same text always maps to the same vector, so search results are stable
    across runs even though no real semantic meaning is captured.
    """

    def __init__(self, dim: int = EMBEDDING_DIM):
        self.dim = dim

    def embed_query(self, text: str):
        seed = int(hashlib.sha256(text.encode("utf-8")).hexdigest(), 16) % (2**32)
        rng = np.random.default_rng(seed)
        vector = rng.normal(size=self.dim)
        vector /= np.linalg.norm(vector)  # unit length, like real embeddings
        return vector.tolist()

    def embed_documents(self, texts):
        return [self.embed_query(t) for t in texts]


def get_embedding_model():
    if TEST_MODE:
        return FakeEmbeddings()
    from langchain_google_genai import GoogleGenerativeAIEmbeddings

    return GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL_NAME, google_api_key=GEMINI_API_KEY)


def get_qdrant_client():
    if TEST_MODE:
        return QdrantClient(":memory:")
    return QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


embedding_model = get_embedding_model()
qdrant_client = get_qdrant_client()


## Recipe data

We start from 5 hand-written sample recipes so the demo output is easy to read. We also
provide `generate_synthetic_recipes(n)`, built with [Faker](https://faker.readthedocs.io/),
to create as many extra mock recipes as you like — useful for testing how the pipeline
behaves at scale.

> **Why mock data instead of a real dataset?** Public recipe datasets exist (for example
> [RecipeNLG](https://recipenlg.cs.put.poznan.pl/) or the
> [Food.com Recipes and Interactions](https://www.kaggle.com/datasets/shuyangli94/food-com-recipes-and-user-interactions)
> dataset on Kaggle, both several GB with 200k–2M+ recipes). They are not bundled in this
> repo because of their size and license terms, and because embedding a real dataset that
> large through a paid API is not something you want happening on a fresh clone. Faker lets
> anyone regenerate a dataset of *any* size — including the 1,000,000-recipe scale test in
> the next section — for free and offline, exercising exactly the same code path a real
> dataset would use.


In [4]:
from faker import Faker

fake = Faker()
Faker.seed(42)

SAMPLE_RECIPES = [
    ("Spicy Thai Green Curry", "A classic Thai green curry with chicken, coconut milk, and fresh basil.", "Thai", "All"),
    ("Hearty Winter Stew", "A rich and comforting beef stew with root vegetables, perfect for a cold day.", "American", "Winter"),
    ("Summer Berry Salad", "A light and refreshing salad with mixed greens, fresh berries, and a vinaigrette dressing.", "Fusion", "Summer"),
    ("Pad Thai", "A popular Thai stir-fried noodle dish with shrimp, tofu, and peanuts.", "Thai", "All"),
    ("Pumpkin Spice Soup", "A creamy and flavorful soup made with roasted pumpkin and warm spices.", "American", "Autumn"),
]

_CUISINES = ["Thai", "American", "Fusion", "Italian", "Mexican", "Indian", "Japanese", "French", "Mediterranean"]
_SEASONS = ["All", "Winter", "Summer", "Autumn", "Spring"]
_MAIN_INGREDIENTS = ["chicken", "tofu", "beef", "salmon", "lentils", "mushroom", "shrimp", "pumpkin", "chickpea"]
_STYLES = ["stir-fried", "slow-cooked", "roasted", "grilled", "one-pot", "baked", "steamed"]


def generate_synthetic_recipes(n: int, seed: int = 42):
    """Generate `n` mock recipes for scale testing. Deterministic given the same seed."""
    local_fake = Faker()
    Faker.seed(seed)
    recipes = []
    for _ in range(n):
        cuisine = local_fake.random_element(_CUISINES)
        season = local_fake.random_element(_SEASONS)
        ingredient = local_fake.random_element(_MAIN_INGREDIENTS)
        style = local_fake.random_element(_STYLES)
        name = f"{style.title()} {ingredient.title()} {local_fake.word().title()}"
        description = (
            f"A {style} {cuisine.lower()} dish built around {ingredient}, "
            f"{local_fake.sentence(nb_words=8)}"
        )
        recipes.append((name, description, cuisine, season))
    return recipes


## Storing and indexing recipes

`setup_database_and_qdrant()` does two jobs:

1. **Store recipe metadata** — in a real Postgres/Neon table when `TEST_MODE = False`, or in
   a simple in-memory Python list when `TEST_MODE = True`.
2. **Index recipes in Qdrant** — every recipe gets a *dense* vector (semantic meaning) and a
   *sparse* vector (TF-IDF keywords), stored together in one point so a single query can
   search both at once.

Note the collection setup below uses `collection_exists()` + `create_collection()` instead of
the older `recreate_collection()`, which is deprecated in current `qdrant-client` releases.


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

# In TEST_MODE this list stands in for the Postgres `recipes` table.
IN_MEMORY_RECIPES = []


def _get_db_connection():
    import psycopg2

    return psycopg2.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)


def _ensure_qdrant_collection():
    if qdrant_client.collection_exists(COLLECTION_NAME):
        qdrant_client.delete_collection(COLLECTION_NAME)

    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "semantic-vector": models.VectorParams(size=EMBEDDING_DIM, distance=models.Distance.COSINE),
        },
        sparse_vectors_config={
            "keyword-vector": models.SparseVectorParams(index=models.SparseIndexParams(on_disk=False)),
        },
    )


def setup_database_and_qdrant(extra_recipes=None):
    """Create/reset storage, insert recipes, and index them into Qdrant.

    `extra_recipes` lets the scale-test cell below add many more mock recipes
    on top of the 5 curated samples.
    """
    recipes_to_load = list(SAMPLE_RECIPES) + list(extra_recipes or [])

    if TEST_MODE:
        IN_MEMORY_RECIPES.clear()
        for i, (name, description, cuisine, season) in enumerate(recipes_to_load, start=1):
            IN_MEMORY_RECIPES.append(
                {"id": i, "name": name, "description": description, "cuisine": cuisine, "season": season}
            )
        rows = [(r["id"], r["name"], r["description"], r["cuisine"], r["season"]) for r in IN_MEMORY_RECIPES]
        print(f"Stored {len(rows)} recipes in memory (TEST_MODE).")
    else:
        conn = _get_db_connection()
        cur = conn.cursor()
        cur.execute(
            """
            CREATE TABLE IF NOT EXISTS recipes (
                id SERIAL PRIMARY KEY,
                name VARCHAR(255) NOT NULL,
                description TEXT,
                cuisine VARCHAR(100),
                season VARCHAR(50)
            );
            """
        )
        cur.execute("TRUNCATE TABLE recipes RESTART IDENTITY;")
        for recipe in recipes_to_load:
            cur.execute(
                "INSERT INTO recipes (name, description, cuisine, season) VALUES (%s, %s, %s, %s)", recipe
            )
        conn.commit()
        cur.execute("SELECT id, name, description, cuisine, season FROM recipes;")
        rows = cur.fetchall()
        cur.close()
        conn.close()
        print(f"Stored {len(rows)} recipes in Postgres.")

    # --- Index everything into Qdrant (dense + sparse in one point each) ---
    _ensure_qdrant_collection()

    all_descriptions = [row[2] for row in rows]
    tfidf_vectorizer.fit(all_descriptions)

    batch_size = 1000
    indexed = 0
    start = time.time()
    for batch_start in range(0, len(rows), batch_size):
        batch = rows[batch_start:batch_start + batch_size]
        points = []
        for recipe_id, name, description, cuisine, season in batch:
            dense_embedding = embedding_model.embed_query(description)
            sparse_vector = tfidf_vectorizer.transform([description])
            points.append(
                models.PointStruct(
                    id=recipe_id,
                    vector={
                        "semantic-vector": dense_embedding,
                        "keyword-vector": models.SparseVector(
                            indices=sparse_vector.indices.tolist(),
                            values=sparse_vector.data.tolist(),
                        ),
                    },
                    payload={"name": name, "description": description, "cuisine": cuisine, "season": season},
                )
            )
        qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
        indexed += len(points)

    elapsed = time.time() - start
    print(f"Indexed {indexed} recipes into Qdrant collection '{COLLECTION_NAME}' in {elapsed:.1f}s.")


## Hybrid search

`hybrid_search()` is the core of this notebook: it embeds the query two ways (dense +
sparse), asks Qdrant to fuse both result lists with **Reciprocal Rank Fusion (RRF)**, and
returns the top matches. `RecipeSearchTool` just wraps this function so a CrewAI agent can
call it.


In [6]:
from crewai.tools import BaseTool


def hybrid_search(query: str, limit: int = 3):
    dense_query_embedding = embedding_model.embed_query(query)

    sparse_query_vector = tfidf_vectorizer.transform([query])

    prefetch_queries = [
        models.Prefetch(
            query=dense_query_embedding,
            using="semantic-vector",
            limit=5,
        ),
        models.Prefetch(
            query=models.SparseVector(
                indices=sparse_query_vector.indices.tolist(),
                values=sparse_query_vector.data.tolist(),
            ),
            using="keyword-vector",
            limit=5,
        ),
    ]

    search_results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_queries,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
        with_payload=True,
    )
    return search_results.points


class RecipeSearchTool(BaseTool):
    name: str = "Recipe Search Tool"
    description: str = "Searches for recipes in the Qdrant database using a query. Use this to find recipes based on user preferences."

    def _run(self, query: str) -> str:
        points = hybrid_search(query)
        if not points:
            return "Found no matching recipes."

        results_str = "Found recipes:\n"
        for point in points:
            payload = point.payload
            results_str += (
                f"- {payload['name']}: {payload['description']} "
                f"(Cuisine: {payload['cuisine']}, Season: {payload['season']})\n"
            )
        return results_str


## The CrewAI agent

One agent, one task, one Gemini-powered `Crew`. The agent's only tool is the
`RecipeSearchTool` defined above — it must use hybrid search to find real recipes rather
than inventing them.

This cell only builds the agent; it needs a real `GEMINI_API_KEY` to actually run
(`TEST_MODE = False`), because CrewAI has to call a live LLM to plan and write the final
answer. In `TEST_MODE` we skip the agent and call `hybrid_search()` directly in the next
section, so you can see search results without needing any API key.


In [7]:
def build_recipe_crew():
    from crewai import LLM, Agent, Crew, Process, Task

    llm = LLM(model=GEMINI_LLM_MODEL, temperature=0.7, api_key=GEMINI_API_KEY)

    recipe_expert = Agent(
        role="Recipe Expert",
        goal="Find the best recipes for the user based on their preferences.",
        backstory="You are an expert in all types of cuisine and know the best recipes for any occasion.",
        verbose=True,
        llm=llm,
        allow_delegation=False,
        tools=[RecipeSearchTool()],
    )

    recipe_task = Task(
        description='Find a recipe based on the query: "{query}"',
        expected_output="A friendly response with the recommended recipe(s) and why they are a good fit.",
        agent=recipe_expert,
    )

    return Crew(agents=[recipe_expert], tasks=[recipe_task], process=Process.sequential)


## Run it

In `TEST_MODE` this cell indexes the 5 sample recipes, runs a couple of hybrid searches
directly, and checks the results with simple `assert` statements — a quick, offline
self-test anyone can run right after cloning the repo.

In live mode it also builds the CrewAI crew and asks you for a query, then prints the
agent's friendly recommendation.


In [8]:
print("--- Setting up storage and Qdrant ---")
setup_database_and_qdrant()
print("--- Setup complete ---\n")

if TEST_MODE:
    # Offline self-test: no API key needed, runs in seconds.
    thai_results = hybrid_search("thai noodles with peanuts")
    assert len(thai_results) > 0, "Expected at least one hybrid search result"
    print("Query: 'thai noodles with peanuts'")
    for point in thai_results:
        print(f"  - {point.payload['name']} (score={point.score:.4f})")

    winter_results = hybrid_search("warm comforting stew for a cold day")
    assert len(winter_results) > 0, "Expected at least one hybrid search result"
    print("\nQuery: 'warm comforting stew for a cold day'")
    for point in winter_results:
        print(f"  - {point.payload['name']} (score={point.score:.4f})")

    print("\nSelf-test passed: hybrid search returned results for both queries.")
    print("Set TEST_MODE = False (and add your API keys) to try the full CrewAI + Gemini experience.")
else:
    recipe_crew = build_recipe_crew()
    user_query = input("\nWhat kind of recipe are you looking for? (e.g., 'best thai recipe for dinner during winter'): ")
    result = recipe_crew.kickoff(inputs={"query": user_query})
    print("\n--- Here's the recommendation ---")
    print(result)


--- Setting up storage and Qdrant ---
Stored 5 recipes in memory (TEST_MODE).
Indexed 5 recipes into Qdrant collection 'food_recipes' in 0.0s.
--- Setup complete ---

Query: 'thai noodles with peanuts'
  - Pad Thai (score=0.8333)
  - Pumpkin Spice Soup (score=0.7500)
  - Spicy Thai Green Curry (score=0.5833)

Query: 'warm comforting stew for a cold day'
  - Pumpkin Spice Soup (score=0.8333)
  - Hearty Winter Stew (score=0.8333)
  - Spicy Thai Green Curry (score=0.2500)

Self-test passed: hybrid search returned results for both queries.
Set TEST_MODE = False (and add your API keys) to try the full CrewAI + Gemini experience.


## Optional: scale test with up to 1,000,000 mock recipes

Everything above works the same whether Qdrant holds 5 recipes or 1,000,000. This cell is
**opt-in** (`RUN_SCALE_TEST = False` by default) so it never slows down a normal run.

Flip `RUN_SCALE_TEST` to `True` and re-run this cell to try it. `SCALE_TEST_SIZE` defaults to
a smaller 5,000 for a quick check; set it to `1_000_000` for the full-scale test described in
this repo's task.

**A few honest numbers from testing this on a modest machine**, so you know what to expect:
indexing ran at roughly 400–450 fake recipes/second, so 20,000 recipes took about 45 seconds
and 1,000,000 would take on the order of **35–40 minutes** and a few GB of RAM. Qdrant itself
prints a warning above 20,000 points that its in-memory `:memory:` mode (used in `TEST_MODE`)
is not meant for large collections — for a real 1,000,000-recipe benchmark, point
`QDRANT_URL`/`QDRANT_API_KEY` at a real Qdrant Cloud cluster or a local Qdrant Docker
container instead of relying on `:memory:` mode.


In [9]:
RUN_SCALE_TEST = False
SCALE_TEST_SIZE = 5_000  # set to 1_000_000 for the full-scale test

if RUN_SCALE_TEST:
    print(f"Generating {SCALE_TEST_SIZE:,} mock recipes with Faker...")
    start = time.time()
    synthetic_recipes = generate_synthetic_recipes(SCALE_TEST_SIZE)
    print(f"Generated in {time.time() - start:.1f}s")

    setup_database_and_qdrant(extra_recipes=synthetic_recipes)

    start = time.time()
    results = hybrid_search("a warm slow-cooked chicken dish for winter", limit=5)
    print(f"\nHybrid search across {SCALE_TEST_SIZE + len(SAMPLE_RECIPES):,} recipes took {time.time() - start:.3f}s")
    for point in results:
        print(f"  - {point.payload['name']} (score={point.score:.4f})")
else:
    print("Scale test skipped (RUN_SCALE_TEST = False). Flip it to True to index up to 1,000,000 mock recipes.")


Scale test skipped (RUN_SCALE_TEST = False). Flip it to True to index up to 1,000,000 mock recipes.
